# QLoRA no Qwen3-4B: ajuste fino para o sentimento em português

Treina um adaptador LoRA sobre o modelo base em 4 bits, com 12 mil avaliações do treino e 1 época, e avalia nos mesmos splits de validação e teste dos outros modelos. Toda a lógica está em `src/` (testada); este notebook só chama os scripts.

**Antes de rodar:**
1. Acelerador **GPU T4** ligado (Ambiente de execução > Alterar o tipo de ambiente de execução).
2. Segredo **`HF_TOKEN`** com permissão de escrita (ícone de chave, com *Acesso ao notebook* ativado), para enviar o adaptador ao Hub.
3. O código novo precisa estar **no GitHub** (`git push`), porque o notebook faz `git clone`.

| Etapa | O que faz | Tempo |
| --- | --- | --- |
| Teste rápido | 20 passos de treino, só para ver se tudo funciona e medir tokens/s | poucos minutos |
| Treino | 750 passos (12.000 exemplos, batch efetivo 16) | **não medido na T4**; o teste rápido mostra a estimativa |
| Validação | 600 avaliações | a medir |
| Teste | 12.618 avaliações | a medir |

Se a estimativa do treino passar de ~2 h, reduza `--train-subset` (por exemplo 8000) e registre isso na comparação.

In [ ]:
import torch

assert torch.cuda.is_available(), "Sem GPU: ligue o acelerador nas configurações do notebook."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/arthurvelloso/sentimento-pt.git"
HUB_ADAPTER = "arthurvelloso/qwen3-4b-sentimento-b2w-qlora"  # adaptador (privado)

if not os.path.exists("sentimento-pt"):
    clone = subprocess.run(
        ["git", "clone", REPO_URL],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )
    if clone.returncode != 0:
        raise RuntimeError("Não foi possível clonar. O repositório está público?")
os.chdir("sentimento-pt")
subprocess.run(["git", "pull", "--ff-only"])
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
assert os.path.exists("src/qlora.py"), "O código mais recente ainda não está no GitHub (git push)."

In [ ]:
# Só o que falta; o torch, o pandas e o scikit-learn já vêm instalados. Não reinstale o torch.
!pip install -q -r requirements-train.txt

In [ ]:
# Lê o token dos segredos da plataforma, sem imprimir nada.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except ImportError:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token carregado:", bool(os.environ.get("HF_TOKEN")))

## 1. Teste rápido
20 passos, sem enviar nada ao Hub e gravando fora de `results/`. Confira: os parâmetros treináveis (cerca de 0,5% do total), a loss caindo e os **tokens/s** (o tempo restante é estimado a partir deles). Se der falta de memória, reduza `micro_batch_size` na config e aumente `grad_accum` na mesma proporção.

In [ ]:
!python -m src.qlora --max-steps 20 --output-dir outputs/smoke --results-path outputs/smoke/train.json

## 2. Treino
Envia o adaptador ao Hub (privado) no fim, para não perdê-lo se a sessão cair. Grava `results/qlora_train.json` com hiperparâmetros, curva de loss, memória e tempo.

In [ ]:
!python -m src.qlora --hub-repo $HUB_ADAPTER

## 3. Avaliação
**3a. Validação** (600 avaliações estratificadas, como nas rodadas do LLM sem ajuste). Confira que a taxa de respostas inválidas é ~0 e que o macro-F1 não ficou abaixo do zero-shot (0,696).

In [ ]:
!python -m src.llm_classify --split val --max-eval 600 --adapter outputs/qlora/adapter

**3b. Teste** (inteiro, 12.618 avaliações). Rode **uma vez**. Se a validação indicar que vai demorar demais na T4, acrescente `--max-eval 3000` (subconjunto estratificado; a análise compara só os exemplos em comum).

In [ ]:
!python -m src.llm_classify --split test --adapter outputs/qlora/adapter

## Levar os resultados de volta
Extraia o `.zip` dentro da pasta `sentimento-pt/` do seu computador (os caminhos já são relativos a ela).

In [ ]:
import glob
import zipfile

wanted = [
    "results/qlora_train.json",
    *glob.glob("results/llm_*qlora*.json"),
    *glob.glob("results/preds_llm_*qlora*.csv"),
    *glob.glob("results/dev/*qlora*"),
]
present = [path for path in wanted if os.path.exists(path)]
with zipfile.ZipFile("resultados_qlora.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in present:
        archive.write(path)
print("no zip:", *present, sep="\n  ")
try:
    from google.colab import files as colab_files

    colab_files.download("resultados_qlora.zip")
except ImportError:
    print("Fora do Colab: baixe o resultados_qlora.zip pelo painel de arquivos.")